## (0) Parameters and Constants

- Service region 20x20 [mile sq.]
- #Demand nodes, $n = 10$ (uniformly distributed)
- Manhattan distance ($l1$)
- Vehicle speed, $v = 20$ [mile/hr], $c_{(i,j)} = \frac{1}{20}(|x_{i}-x_{j}|+|y_{i}-y_{j}|)$ [hr]
- $s_{0}=0.5$ [hr], $s_{1} = 0$ 
- $q_{i} \in \{1,2,3,4,5\}$ [packges/hr]
- Maximum carrying capacity, $Q = 20$ [packages]


In [7]:
no_dock = 0 
no_layer = 1
no_demand_node = 5
truck_capacity = 20
fixed_setup_time = 0.5
truck_speed = 20

constant_dict = { 'truck_capacity' : truck_capacity,'fixed_setup_time' : fixed_setup_time,
                    'truck_speed': truck_speed,'max_vehicles':6}

In [8]:
# Create visualization instance
docking,customers,depot,depot_s,depot_t,all_depot,nodes,arcs=vis_sol.create_nodes(no_dock,no_demand_node)

[] ['customer_1', 'customer_2', 'customer_3', 'customer_4', 'customer_5'] ['depot'] ['customer_4', 'customer_1', 'depot', 'customer_2', 'customer_3', 'customer_5']


In [376]:
## SKIP THIS IF DON'T WANT TO USE NEW RANDOM MAP

# Random distance matrix
distance_matrix, node_position = rand_dis_mat(nodes)
# Random customer demand
_depot_demand = pd.Series([0,0], index =all_depot)
customer_demand = random_customer_demand(customers).append(_depot_demand)
# Visualization 
color_list = vis_sol.create_color_list(list(nodes_position.keys()))
node_trace = vis_sol.create_node_trace(list(nodes_position.values()),list(nodes_position.keys()),
                                 Color=color_list)

### Save Instance to pickle

In [377]:
instance_name='DemoInstanceCus{0}_1'.format(no_demand_node);print(instance_name);
instance_data = {'distance_matrix':distance_matrix,
               'node_position':node_position,
               'node_trace':node_trace,
                'customer_demand_df':customer_demand}
# instance_data

DemoInstanceCus5_1


In [378]:
with open('./Instances/%s.pickle'%instance_name,'wb') as f1:
    pk.dump(instance_data,f1)
    print(instance_name+" is saved succuessfully!")

DemoInstanceCus5_1 is saved succuessfully!


### Import instance

In [11]:
instance_name='DemoInstanceCus{0}_1'.format(no_demand_node);print(instance_name);
with open('./Instances/%s.pickle'%instance_name,'rb') as f1:
    r_instance = pk.load(f1)

distance_matrix = r_instance['distance_matrix']
node_trace = r_instance['node_trace']
customer_demand = r_instance['customer_demand_df']

DemoInstanceCus5_1


In [16]:
vis_sol.plot_solution([['customer_1,customer_2']],node_trace, None,"Demo Instance", customer_demand)

['customer_1', 'customer_2', 'T']


## (1) Initial solution set $\mathcal{R}^{+}$

In [381]:
initializer = InitialRouteGenerator(no_layer,no_dock,no_demand_node,customer_demand,constant_dict,distance_matrix)
# Generate node
(docking,customers,depot,depot_s,
 depot_t,all_depot,nodes,arcs) = initializer.generateNodeSet()

[] ['customer_1', 'customer_2', 'customer_3', 'customer_4', 'customer_5'] ['depot'] ['customer_3', 'customer_1', 'depot', 'customer_2', 'customer_4', 'customer_5']


In [382]:
# Generate initial feasilble routes
row_labels = ['lr','m']+all_depot+customers+arcs
x = pd.DataFrame(data =row_labels,columns=['labels'])
initializer.generateRoutes(initRouteDf=x,
                           truck_cap_limit=truck_capacity,
                           max_visited_nodes=4, max_vehicles_per_route=3)
x = x.set_index('labels')

nbInitRoute is set to 205
progress: 0 / 205
progress: 41 / 205
progress: 82 / 205
progress: 123 / 205
progress: 164 / 205
#Feasible Cols: 457
Elased-time: 0.5013868808746338


In [383]:
# Reformatting
feasibleCols = x.shape[1]
init_route = initializer.generateBasicInitialPatterns(feasibleCols,initRouteDf=x)
init_route.rename(index=lambda x:'route[%d]'%x,inplace=True)

In [384]:
x.head(20)

,route[0],route[1],route[2],route[3],route[4],route[5],route[6],route[7],route[8],route[9],...,route[447],route[448],route[449],route[450],route[451],route[452],route[453],route[454],route[455],route[456]
labels,,,,,,,,,,,,,,,,,,,,,
lr,1.777405,1.777405,1.777405,0.808164,0.808164,0.808164,1.730476,1.730476,1.730476,1.718187,...,2.170864,2.170864,2.852985,2.852985,2.08621,2.08621,2.852985,2.852985,2.08621,2.08621
m,1.000000,2.000000,3.000000,1.000000,2.000000,3.000000,1.000000,2.000000,3.000000,1.000000,...,2.000000,3.000000,2.000000,3.000000,2.00000,3.00000,2.000000,3.000000,2.00000,3.00000
depot_s,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.00000,1.00000,1.000000,1.000000,1.00000,1.00000
depot_t,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.00000,1.00000,1.000000,1.000000,1.00000,1.00000
customer_1,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.000000,0.000000,0.00000,0.00000
customer_2,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,...,1.000000,1.000000,1.000000,1.000000,1.00000,1.00000,1.000000,1.000000,1.00000,1.00000
customer_3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,0.000000,...,1.000000,1.000000,1.000000,1.000000,1.00000,1.00000,1.000000,1.000000,1.00000,1.00000
customer_4,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.00000,1.00000,1.000000,1.000000,1.00000,1.00000
customer_5,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,1.000000,1.000000,1.000000,1.000000,1.00000,1.00000,1.000000,1.000000,1.00000,1.00000


### Plot sample of route

In [386]:
sample_r = x['route[89]']
sample_arcs = sample_r[sample_r.index.isin(arcs)][sample_r==1]
reformatted_arcs = [merge_depot_arcs_var([t],depot,depot_s,depot_t) for t in sample_arcs.index.to_list()]
print('mr:',sample_r['m'], 'lr:',sample_r['lr'])
vis_sol.plot_solution(reformatted_arcs,node_trace, None,"Demo Instance", customer_demand)

mr: 3.0 lr: 2.503583148490494
['customer_3', 'customer_1', 'T']
['depot', 'customer_3', 'T']
['customer_1', 'customer_2', 'T']
['customer_2', 'depot', 'T']


In [387]:
# feasibleCols = x.shape[1]
# init_route = initializer.generateBasicInitialPatterns(feasibleCols,initRouteDf=x)
# init_route.rename(index=lambda x:'route[%d]'%x,inplace=True)
# route_idx_labels = x.index.values

In [388]:
# init_route

## (2) Phase I: Minimize #vehicle

In [389]:
phaseI_model = phaseIModel(init_route, initializer,
                 distance_matrix,constant_dict)

In [390]:
phaseI_model.buildModel()

Finish generating variables!
Finish generating constrains!
Finish generating objective!


In [391]:
phaseI_model.solveModel(np.inf,0.1)

Changed value of parameter PoolSearchMode to 2
   Prev: 0  Min: 0  Max: 2  Default: 0
Parameter TImeLimit unchanged
   Value: inf  Min: 0.0  Max: inf  Default: inf
Changed value of parameter MIPGap to 0.1
   Prev: 0.0001  Min: 0.0  Max: inf  Default: 0.0001
Gurobi Optimizer version 9.1.1 build v9.1.1rc0 (mac64)
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads
Optimize a model with 5 rows, 457 columns and 1509 nonzeros
Model fingerprint: 0x754433b6
Variable types: 0 continuous, 457 integer (457 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 3e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]
Found heuristic solution: objective 5.0000000
Presolve time: 0.01s
Presolved: 5 rows, 457 columns, 1509 nonzeros
Variable types: 0 continuous, 457 integer (457 binary)

Root relaxation: objective 2.000000e+00, 8 iterations, 0.00 seconds

    Nodes    |    Current Node    |     Objective Bounds      |     

In [392]:
sol_obj_phaseI = pd.Series(phaseI_model.model.getVars())
sol_vec_phaseI = pd.DataFrame(index = sol_obj_phaseI.apply(lambda x:x.VarName))
sol_vec_phaseI['value'] = sol_obj_phaseI.apply(lambda x:x.X).values
optimal_routes_phaseI = sol_vec_phaseI.loc[sol_vec_phaseI['value']==1]
optimal_routes_phaseI.index

Index(['route[57]', 'route[111]'], dtype='object')

In [393]:
reformatted_arcs = prep_for_plot(x,optimal_routes_phaseI,arcs)
print(x.loc[['m','lr']][optimal_routes_phaseI.index])
vis_sol.plot_solution(reformatted_arcs,node_trace, None,"Demo Instance", customer_demand)

        route[57]  route[111]
labels                       
m        1.000000     1.00000
lr       1.979967     1.86206
['depot', 'customer_3', 'T']
['customer_3', 'customer_4', 'T']
['customer_4', 'depot', 'T']
['depot', 'customer_1', 'T']
['customer_1', 'customer_2', 'T']
['customer_5', 'depot', 'T']
['customer_2', 'customer_5', 'T']


## (3) Phase II: Cycle time VRP
(Set partitioning)

In [394]:
phaseII_model = phaseIIModel(init_route, initializer,
                 distance_matrix,constant_dict)

In [395]:
phaseII_model.buildModel()

Finish generating variables!
Finish generating constrains!
Finish generating cost vector!....Elapsed-time: 0.5797119140625
Finish generating objective!


In [396]:
phaseII_model.route_cost

route[0]       1.527405
route[1]       1.083054
route[2]       0.934937
route[3]       0.558164
route[4]       0.356123
                ...    
route[452]    12.469845
route[453]    20.403062
route[454]    17.312329
route[455]    14.680751
route[456]    12.420691
Length: 457, dtype: float64

In [397]:
# phaseII_model.route_cost['route[0]']

In [398]:
phaseII_model.solveModel(120,0.1)

Changed value of parameter PoolSearchMode to 2
   Prev: 0  Min: 0  Max: 2  Default: 0
Changed value of parameter TImeLimit to 120.0
   Prev: inf  Min: 0.0  Max: inf  Default: inf
Changed value of parameter MIPGap to 0.1
   Prev: 0.0001  Min: 0.0  Max: inf  Default: 0.0001
Gurobi Optimizer version 9.1.1 build v9.1.1rc0 (mac64)
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads
Optimize a model with 6 rows, 457 columns and 1966 nonzeros
Model fingerprint: 0x60bdc0c4
Variable types: 0 continuous, 457 integer (457 binary)
Coefficient statistics:
  Matrix range     [1e+00, 3e+00]
  Objective range  [3e-01, 3e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 6e+00]
Presolve time: 0.00s
Presolved: 6 rows, 457 columns, 1966 nonzeros
Variable types: 0 continuous, 457 integer (457 binary)
Found heuristic solution: objective 59.6688995
Found heuristic solution: objective 26.4684985
Found heuristic solution: objective 18.5967121

Root relaxation: objective 

In [399]:
sol_obj_phaseII = pd.Series(phaseII_model.model.getVars())
sol_vec_phaseII = pd.DataFrame(index = sol_obj_phaseII.apply(lambda x:x.VarName))
sol_vec_phaseII['value'] = sol_obj_phaseII.apply(lambda x:x.X).values
optimal_routes_phaseII = sol_vec_phaseII.loc[sol_vec_phaseII['value']==1]
optimal_routes_phaseII.index

Index(['route[20]', 'route[228]'], dtype='object')

In [400]:
reformatted_arcs_II = prep_for_plot(x,optimal_routes_phaseII,arcs)
print(x.loc[['m','lr']][optimal_routes_phaseII.index])
vis_sol.plot_solution(reformatted_arcs_II,node_trace, None,"Demo Instance", customer_demand)

        route[20]  route[228]
labels                       
m        3.000000    3.000000
lr       1.777405    1.979967
['customer_1', 'depot', 'T']
['customer_2', 'customer_1', 'T']
['depot', 'customer_2', 'T']
['customer_3', 'depot', 'T']
['customer_4', 'customer_3', 'T']
['depot', 'customer_5', 'T']
['customer_5', 'customer_4', 'T']


In [401]:
distance_matrix[('customer_2','depot')]

3.0816386272185206

In [402]:
total_cost_phaseI_sol = phaseII_model.route_cost[optimal_routes_phaseI.index].sum()
total_cost_phaseII_sol = phaseII_model.route_cost[optimal_routes_phaseII.index].sum()
print('cost of phase I sol:',total_cost_phaseI_sol,
                              ", Avg mins:",total_cost_phaseI_sol*60/customer_demand.sum(),
     '\ncost of phase II sol:', total_cost_phaseII_sol,
                                  ", Avg mins:",total_cost_phaseII_sol*60/customer_demand.sum())

cost of phase I sol: 26.087605434513364 , Avg mins: 111.80402329077155 
cost of phase II sol: 11.81362830466631 , Avg mins: 50.62983559142704


In [ ]:
# ans[optimal_routes_phaseII.index].iloc[phaseII_model.arcs_index]

In [ ]:
# x = ans[optimal_routes_phaseII.index[0]]
# x.loc[x>0]

In [ ]:
# customer_demand.sum()

## (4) DP subproblem
State:= ()

In [403]:
phaseII_model.solveRelaxedModel()

Gurobi Optimizer version 9.1.1 build v9.1.1rc0 (mac64)
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads
Optimize a model with 6 rows, 457 columns and 1966 nonzeros
Model fingerprint: 0x133aa9a0
Coefficient statistics:
  Matrix range     [1e+00, 3e+00]
  Objective range  [3e-01, 3e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 6e+00]
Presolve time: 0.01s
Presolved: 6 rows, 457 columns, 1966 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    5.7755182e-01   5.000000e+00   0.000000e+00      0s
      15    1.1549976e+01   0.000000e+00   0.000000e+00      0s

Solved in 15 iterations and 0.01 seconds
Optimal objective  1.154997600e+01


In [472]:
duals_vect = pd.DataFrame(phaseII_model.getDuals(), index = phaseII_model.customers + ['m'])
duals_vect

,0
customer_1,1.946862
customer_2,1.118050
customer_3,5.127934
customer_4,5.086972
customer_5,1.893129
m,-0.559887


In [474]:
distance_matrix

{('customer_1', 'customer_3'): 14.992853661630734,
 ('customer_3', 'customer_1'): 14.992853661630734,
 ('depot', 'customer_3'): 12.304756752027492,
 ('customer_3', 'depot'): 12.304756752027492,
 ('depot', 'customer_1'): 12.774052556151645,
 ('customer_1', 'depot'): 12.774052556151645,
 ('customer_2', 'customer_3'): 11.34797647844474,
 ('customer_3', 'customer_2'): 11.34797647844474,
 ('customer_2', 'customer_1'): 9.692413928933124,
 ('customer_1', 'customer_2'): 9.692413928933124,
 ('customer_2', 'depot'): 3.0816386272185206,
 ('depot', 'customer_2'): 3.0816386272185206,
 ('customer_4', 'customer_3'): 5.1127094862779865,
 ('customer_3', 'customer_4'): 5.1127094862779865,
 ('customer_4', 'customer_1'): 14.499276756380642,
 ('customer_1', 'customer_4'): 14.499276756380642,
 ('customer_4', 'depot'): 12.181871954271454,
 ('depot', 'customer_4'): 12.181871954271454,
 ('customer_4', 'customer_2'): 11.225091680688703,
 ('customer_2', 'customer_4'): 11.225091680688703,
 ('customer_5', 'custome

In [424]:
m_veh = 2

In [502]:
#Initialization
s_0 = SPPState('depot_s', 0,0,0,0)
prev = dict(); reward = dict()
prev[s_0] = -1
reward[s_0] = m_veh*duals_vect.loc['m',0]
L = [s_0]
P = []

arcs_ss = pd.Series(phaseII_model.arcs)

return_arcs = arcs_ss[arcs_ss.str.contains(depot_t[0])]            
return_arcs_cost_dict = pd.Series(index = return_arcs, data = return_arcs.apply(lambda x: distance_matrix[tuple(x.replace('_t','').split(','))]/constant_dict['truck_speed']).values).to_dict()

What needed in DP:
- dual_vectors
- distance_matrix

In [503]:
label_counter = 0
print('DP subproblem for m =', m_veh)
while len(L) != 0:
    currState = L.pop(0)
    if currState.resNode == 'depot_s': delta_plus = arcs_ss[arcs_ss.str.contains(currState.resNode)].apply(lambda x: tuple(x.replace('_s','').split(',')))
    else: delta_plus = arcs_ss[arcs_ss.str.contains(currState.resNode+',')].apply(lambda x: tuple(x.split(',')))
    for arc in delta_plus.to_list():
        label_counter +=1; print(label_counter,currState.resNode, arc)
        if arc[-1] == depot_t[0]:
            ext_node = depot[0]
            formatted_arc = (currState.resNode,ext_node)
            return_cost = 0
        else:
            ext_node = arc[-1]
            formatted_arc = arc
            return_cost = return_arcs_cost_dict[','.join([arc[-1]]+depot_t)]

        extState = SPPState(arc[-1], currState.accDemand+customer_demand[arc[-1]], 
                           currState.accDistance+(phaseII_model.distance_matrix[formatted_arc]/20), 
                            currState.nodeVisited+1,label_counter)
        print(currState.label, currState.resNode,reward[currState])

        if checkFeasibility(extState, phaseII_model.vehicle_capacity,m_veh,return_cost):
            prev[extState] = currState
            #Can be improved!
            reward[extState] = reward[currState]+transitionReward(currState,extState,duals_vect,m_veh)
            L.append(extState)
#             if reward[extState] > 0: break
        else: continue
    P.append(currState)
    #     print(checkFeasibility(extState, phaseII_model.vehicle_capacity, phaseII_model.max_vehicles,return_cost))


DP subproblem for m = 2
1 depot_s ('depot', 'customer_3')
0 depot_s -1.119773023916526
resNode: customer_3 deliver_cap: 4.921902700810997 cap_limit: 40
2 depot_s ('depot', 'customer_1')
0 depot_s -1.119773023916526
resNode: customer_1 deliver_cap: 1.2774052556151645 cap_limit: 40
3 depot_s ('depot', 'customer_2')
0 depot_s -1.119773023916526
resNode: customer_2 deliver_cap: 0.30816386272185203 cap_limit: 40
4 depot_s ('depot', 'customer_4')
0 depot_s -1.119773023916526
resNode: customer_4 deliver_cap: 4.872748781708582 cap_limit: 40
5 depot_s ('depot', 'customer_5')
0 depot_s -1.119773023916526
resNode: customer_5 deliver_cap: 1.146301657475083 cap_limit: 40
6 customer_3 ('customer_3', 'customer_1')
1 customer_3 -0.18326620272671934
resNode: customer_1 deliver_cap: 10.017915742452468 cap_limit: 40
7 customer_3 ('customer_3', 'depot_t')
1 customer_3 -0.18326620272671934
resNode: depot_t deliver_cap: 4.921902700810997 cap_limit: 40
8 customer_3 ('customer_3', 'customer_2')
1 customer_3 -

174 customer_3 ('customer_3', 'customer_4')
41 customer_3 -2.952652697249306
resNode: customer_4 deliver_cap: 31.859805732106324 cap_limit: 40
175 customer_3 ('customer_3', 'customer_5')
41 customer_3 -2.952652697249306
resNode: customer_5 deliver_cap: 27.867945981288766 cap_limit: 40
176 customer_1 ('customer_1', 'customer_3')
42 customer_1 -2.6510010379410094
resNode: customer_3 deliver_cap: 38.48932971542382 cap_limit: 40
177 customer_1 ('customer_1', 'depot_t')
42 customer_1 -2.6510010379410094
resNode: depot_t deliver_cap: 20.110857997876998 cap_limit: 40
178 customer_1 ('customer_1', 'customer_2')
42 customer_1 -2.6510010379410094
resNode: customer_2 deliver_cap: 22.34539777541888 cap_limit: 40
179 customer_1 ('customer_1', 'customer_4')
42 customer_1 -2.6510010379410094
resNode: customer_4 deliver_cap: 38.08862960846984 cap_limit: 40
180 customer_1 ('customer_1', 'customer_5')
42 customer_1 -2.6510010379410094
resNode: customer_5 deliver_cap: 29.21102815948058 cap_limit: 40
181 

resNode: customer_5 deliver_cap: 20.914653695939784 cap_limit: 40
341 customer_2 ('customer_2', 'customer_3')
84 customer_2 -0.9847286655685055
resNode: customer_3 deliver_cap: 24.59227760953408 cap_limit: 40
342 customer_2 ('customer_2', 'customer_1')
84 customer_2 -0.9847286655685055
resNode: customer_1 deliver_cap: 16.799400965788244 cap_limit: 40
343 customer_2 ('customer_2', 'depot_t')
84 customer_2 -0.9847286655685055
resNode: depot_t deliver_cap: 8.584038184744335 cap_limit: 40
344 customer_2 ('customer_2', 'customer_4')
84 customer_2 -0.9847286655685055
resNode: customer_4 deliver_cap: 24.469392811778043 cap_limit: 40
345 customer_2 ('customer_2', 'customer_5')
84 customer_2 -0.9847286655685055
resNode: customer_5 deliver_cap: 15.153275001194297 cap_limit: 40
346 customer_5 ('customer_5', 'customer_3')
85 customer_5 -4.866587004335929
resNode: customer_3 deliver_cap: 29.48829486125786 cap_limit: 40
347 customer_5 ('customer_5', 'customer_1')
85 customer_5 -4.866587004335929
res

resNode: customer_5 deliver_cap: 23.67947055406154 cap_limit: 40
466 customer_5 ('customer_5', 'customer_3')
115 customer_5 -4.779862716963002
resNode: customer_3 deliver_cap: 34.79001497658764 cap_limit: 40
467 customer_5 ('customer_5', 'customer_1')
115 customer_5 -4.779862716963002
resNode: customer_1 deliver_cap: 29.03898276527489 cap_limit: 40
468 customer_5 ('customer_5', 'depot_t')
115 customer_5 -4.779862716963002
resNode: depot_t deliver_cap: 14.765708102432992 cap_limit: 40
469 customer_5 ('customer_5', 'customer_2')
115 customer_5 -4.779862716963002
resNode: customer_2 deliver_cap: 17.377341707498992 cap_limit: 40
470 customer_5 ('customer_5', 'customer_4')
115 customer_5 -4.779862716963002
resNode: customer_4 deliver_cap: 34.59339930017798 cap_limit: 40
471 customer_3 ('customer_3', 'customer_1')
116 customer_3 -4.640497371079026
resNode: customer_1 deliver_cap: 27.891530103502408 cap_limit: 40
472 customer_3 ('customer_3', 'depot_t')
116 customer_3 -4.640497371079026
resNo

156 customer_3 -11.607783047478277
resNode: depot_t deliver_cap: 30.321378786531792 cap_limit: 40
633 customer_3 ('customer_3', 'customer_2')
156 customer_3 -11.607783047478277
resNode: customer_2 deliver_cap: 34.52218875968464 cap_limit: 40
634 customer_3 ('customer_3', 'customer_4')
156 customer_3 -11.607783047478277
resNode: customer_4 deliver_cap: 45.94280758310987 cap_limit: 40
635 customer_3 ('customer_3', 'customer_5')
156 customer_3 -11.607783047478277
resNode: customer_5 deliver_cap: 42.449930301144505 cap_limit: 40
636 customer_2 ('customer_2', 'customer_3')
158 customer_2 -4.18202968828421
resNode: customer_3 deliver_cap: 36.67966187534589 cap_limit: 40
637 customer_2 ('customer_2', 'customer_1')
158 customer_2 -4.18202968828421
resNode: customer_1 deliver_cap: 26.201611029369303 cap_limit: 40
638 customer_2 ('customer_2', 'depot_t')
158 customer_2 -4.18202968828421
resNode: depot_t deliver_cap: 16.14171990044495 cap_limit: 40
639 customer_2 ('customer_2', 'customer_4')
158 

...

8754 customer_2 -10.613611397810203
resNode: customer_5 deliver_cap: 48.784186133797704 cap_limit: 40
10256 customer_2 ('customer_2', 'customer_3')
8759 customer_2 -9.524646666605708
resNode: customer_3 deliver_cap: 54.004342967416875 cap_limit: 40
10257 customer_2 ('customer_2', 'customer_1')
8759 customer_2 -9.524646666605708
resNode: customer_1 deliver_cap: 44.89536478872501 cap_limit: 40
10258 customer_2 ('customer_2', 'depot_t')
8759 customer_2 -9.524646666605708
resNode: depot_t deliver_cap: 26.746598691330536 cap_limit: 40
10259 customer_2 ('customer_2', 'customer_4')
8759 customer_2 -9.524646666605708
resNode: customer_4 deliver_cap: 53.7585733719048 cap_limit: 40
10260 customer_2 ('customer_2', 'customer_5')
8759 customer_2 -9.524646666605708
resNode: customer_5 deliver_cap: 35.126337750737314 cap_limit: 40
10261 customer_2 ('customer_2', 'customer_3')
8763 customer_2 -8.829313059886738
resNode: customer_3 deliver_cap: 68.92086462436917 cap_limit: 40
10262 customer_2 ('custome

resNode: depot_t deliver_cap: 35.31077906786751 cap_limit: 40
10464 customer_5 ('customer_5', 'customer_2')
9185 customer_5 -25.713060036073202
resNode: customer_2 deliver_cap: 39.42662133794201 cap_limit: 40
10465 customer_5 ('customer_5', 'customer_4')
9185 customer_5 -25.713060036073202
resNode: customer_4 deliver_cap: 62.16858514630898 cap_limit: 40
10466 customer_2 ('customer_2', 'customer_3')
9203 customer_2 -19.533997429306105
resNode: customer_3 deliver_cap: 65.447221167274 cap_limit: 40
10467 customer_2 ('customer_2', 'customer_1')
9203 customer_2 -19.533997429306105
resNode: customer_1 deliver_cap: 51.64601688658672 cap_limit: 40
10468 customer_2 ('customer_2', 'depot_t')
9203 customer_2 -19.533997429306105
resNode: depot_t deliver_cap: 36.680526824211384 cap_limit: 40
10469 customer_2 ('customer_2', 'customer_4')
9203 customer_2 -19.533997429306105
resNode: customer_4 deliver_cap: 65.26289397063994 cap_limit: 40
10470 customer_2 ('customer_2', 'customer_5')
9203 customer_2 -

9770 customer_5 -14.830346301963019
resNode: customer_3 deliver_cap: 68.47946435400007 cap_limit: 40
10667 customer_5 ('customer_5', 'customer_1')
9770 customer_5 -14.830346301963019
resNode: customer_1 deliver_cap: 61.16676265842284 cap_limit: 40
10668 customer_5 ('customer_5', 'depot_t')
9770 customer_5 -14.830346301963019
resNode: depot_t deliver_cap: 38.18821507832049 cap_limit: 40
10669 customer_5 ('customer_5', 'customer_2')
9770 customer_5 -14.830346301963019
resNode: customer_2 deliver_cap: 42.32872710355407 cap_limit: 40
10670 customer_5 ('customer_5', 'customer_4')
9770 customer_5 -14.830346301963019
resNode: customer_4 deliver_cap: 68.18454083938556 cap_limit: 40
10671 customer_2 ('customer_2', 'customer_3')
9779 customer_2 -13.643484407277501
resNode: customer_3 deliver_cap: 68.47946435400007 cap_limit: 40
10672 customer_2 ('customer_2', 'customer_1')
9779 customer_2 -13.643484407277501
resNode: customer_1 deliver_cap: 58.67395122709323 cap_limit: 40
10673 customer_2 ('cust

10214 customer_2 -17.33355476683649
resNode: customer_5 deliver_cap: 45.00053265097577 cap_limit: 40
10841 customer_2 ('customer_2', 'customer_3')
10224 customer_2 -14.412977757203212
resNode: customer_3 deliver_cap: 70.51117161788905 cap_limit: 40
10842 customer_2 ('customer_2', 'customer_1')
10224 customer_2 -14.412977757203212
resNode: customer_1 deliver_cap: 60.45169508299608 cap_limit: 40
10843 customer_2 ('customer_2', 'depot_t')
10224 customer_2 -14.412977757203212
resNode: depot_t deliver_cap: 38.188215078320496 cap_limit: 40
10844 customer_2 ('customer_2', 'customer_4')
10224 customer_2 -14.412977757203212
resNode: customer_4 deliver_cap: 70.21624810327455 cap_limit: 40
10845 customer_2 ('customer_2', 'customer_5')
10224 customer_2 -14.412977757203212
resNode: customer_5 deliver_cap: 47.85756535787357 cap_limit: 40
10846 customer_2 ('customer_2', 'customer_3')
10243 customer_2 -8.688287986553497
resNode: customer_3 deliver_cap: 63.913558230186574 cap_limit: 40
10847 customer_2

resNode: customer_3 deliver_cap: 72.2778633845335 cap_limit: 40
11047 customer_2 ('customer_2', 'customer_1')
10819 customer_2 -22.685200596940987
resNode: customer_1 deliver_cap: 62.29962635846327 cap_limit: 40
11048 customer_2 ('customer_2', 'depot_t')
10819 customer_2 -22.685200596940987
resNode: depot_t deliver_cap: 39.113755909591745 cap_limit: 40
11049 customer_2 ('customer_2', 'customer_4')
10819 customer_2 -22.685200596940987
resNode: customer_4 deliver_cap: 71.9706513901434 cap_limit: 40
11050 customer_2 ('customer_2', 'customer_5')
10819 customer_2 -22.685200596940987
resNode: customer_5 deliver_cap: 48.68035686368405 cap_limit: 40
11051 customer_2 ('customer_2', 'customer_3')
10834 customer_2 -19.215708654648346
resNode: customer_3 deliver_cap: 72.27786338453349 cap_limit: 40
11052 customer_2 ('customer_2', 'customer_1')
10834 customer_2 -19.215708654648346
resNode: customer_1 deliver_cap: 62.29962635846327 cap_limit: 40
11053 customer_2 ('customer_2', 'depot_t')
10834 custo

In [501]:
reward[s_0]

-1.119773023916526

In [505]:
a = pd.Series(reward)

In [506]:
b = a[a>0]
b

<__main__.SPPState object at 0x7f7f6b98a100>    0.263652
<__main__.SPPState object at 0x7f7f59ec8f40>    0.263652
<__main__.SPPState object at 0x7f7f6b984100>    0.013681
<__main__.SPPState object at 0x7f7f78e8b760>    0.013681
dtype: float64

In [465]:
def reconstructPath(_last_state, _prev_dict):
    curr_state = _last_state
    path = [curr_state.resNode]
    while curr_state.resNode != depot_s[0]:
#         print('currNode:',curr_state.resNode )
        prev_state = _prev_dict[curr_state]
        path = [prev_state.resNode] + path
        curr_state = prev_state
#         print('nextNode:',curr_state.resNode )
    return path

In [507]:
for s in b.index:
    print(reconstructPath(s,prev), s.label)

['depot_s', 'customer_2', 'customer_1'] 17
['depot_s', 'customer_2', 'customer_1', 'depot_t'] 77
['depot_s', 'customer_2', 'customer_5', 'customer_2'] 89
['depot_s', 'customer_2', 'customer_5', 'customer_2', 'depot_t'] 363


In [407]:
class SPPState:
    def __init__(self, _resNode, _accDemand, _accDistance, _nodeVisited, _label):
        self.resNode = _resNode
        self.accDemand = _accDemand
        self.accDistance = _accDistance
        self.nodeVisited = _nodeVisited
        self.label = _label
        
    def checkFeasibility(self, _total_demand, _max_vehicle_per_route, _return_cost):
        deliver_cap = _state.accDemand*(_state.accDistance+_return_cost)
        if deliver_cap <= _total_demand*_max_vehicle_per_route: return True
        else: return False

def checkFeasibility(_state, _total_demand, _max_vehicle_per_route,_return_cost):
    deliver_cap = _state.accDemand*(_state.accDistance+_return_cost)
    limit_cap = _total_demand*_max_vehicle_per_route
    print('resNode:',_state.resNode,'deliver_cap:',deliver_cap,'cap_limit:',limit_cap )
    if deliver_cap <=limit_cap: return True
    else: return False
    

    
        

In [230]:
arcs_ss[arcs_ss.str.contains(currState.resNode)].apply(lambda x: tuple(x.split(',')))

11     (depot_s, customer_9)
29    (depot_s, customer_10)
45     (depot_s, customer_3)
59     (depot_s, customer_8)
71     (depot_s, customer_1)
81     (depot_s, customer_7)
90     (depot_s, customer_2)
92     (depot_s, customer_4)
94     (depot_s, customer_6)
96     (depot_s, customer_5)
dtype: object

In [494]:
phaseII_model.route_cost[(phaseII_model.route_cost>1.68) & (phaseII_model.route_cost<1.683)]

route[19]    1.681487
dtype: float64

In [500]:
# phaseII_model.init_routes_df.set_index('labels')['route[19]']

## (XX) Class and functions

In [2]:
MODULE_DIR = '/Users/ravitpichayavet/Documents/GaTechOR/GraduateResearch/CTC_CVRP/Modules'
GUROBI_LICENSE_DIR = '/Users/ravitpichayavet/gurobi.lic'
# INSTANCE_PATH = MODULE_DIR+'Combine_variables/Experiment/'

import sys
sys.path.insert(0,MODULE_DIR)

import visualize_sol as vis_sol
import initialize_path as init_path
import random_instance as rand_inst
import utility as util
import branch_and_price as bnp
import model as md

import numpy as np
from gurobipy import *
import os
os.environ['GRB_LICENSE_FILE'] = GUROBI_LICENSE_DIR


from itertools import combinations,permutations 
import random 
import pandas as pd
import itertools
import plotly.graph_objects as go
import networkx as nx
import plotly.offline as py 
import pickle as pk
import nltk
import time
import copy

from matplotlib import pyplot as plt
from sklearn.datasets.samples_generator import make_blobs
from sklearn.cluster import KMeans
from scipy.spatial import distance
import logging
# logging.basicConfig(filename='myapp.log',filemode='w',format='%(message)s',level=logging.INFO)
# print = logging.info

### Create instances

In [3]:
def rand_dis_mat(nodes, service_region_len = 20):
    # random the nodes position
    no_nodes = len(nodes)
    _distance_matrix=dict()
    _nodes_position = dict()
    rand_seed = np.random.uniform(low=-1.5,high=0)
    for i in range(no_nodes):
        if nodes[i]!=depot[0]:
            _nodes_position[nodes[i]] = list(np.random.rand(2)*service_region_len)
#     print(nodes_position)
    _nodes_position[depot[0]] = [service_region_len/2,service_region_len/2]
#     print(nodes_position)
    for i in range(no_nodes):
        for j in range(no_nodes):
            if i>j:
                point_i = np.array(_nodes_position[nodes[i]])
                point_j = np.array(_nodes_position[nodes[j]])
                t_dist_rand = np.linalg.norm(point_i-point_j,ord=1)
                _distance_matrix[(nodes[i],nodes[j])] = t_dist_rand
                _distance_matrix[(nodes[j],nodes[i])] = t_dist_rand
    return _distance_matrix, _nodes_position

def random_customer_demand(customers):
    demand = np.random.randint(1,6,len(customers)).tolist()
    customer_demand = pd.Series(demand,index = customers)
    return customer_demand

def merge_depot_arcs_var(a_var,depot,depot_s,depot_t):
    '''INPUT: ['depot_s,customer_1,T','customer_2,depot_t,T']
    OUTPUT: ['depot,customer_1,T','customer_2,depot,T']'''
    new_a_var =[]
    for a in a_var:
        v = a.split(',')
        if len(v)>1:
            if v[0]==depot_s[0] or v[0]==depot_t[0]:v[0]=depot[0]
            if v[1]==depot_s[0] or v[1]==depot_t[0]:v[1]=depot[0]
        else:
            if v[0]==depot_s[0] or v[0]==depot_t[0]:v[0] = depot[0]
        new_a_var.append(','.join(v))
    return new_a_var  

def prep_for_plot(route_df,optimal_route_df,_arcs):
    reformatted_arcs=[]
    for idx in optimal_route_df.index:
        sample_r = route_df.loc[:][idx]
        sample_arcs = sample_r[sample_r.index.isin(_arcs)][sample_r==1]
        reformatted_arcs += [merge_depot_arcs_var([t],depot,depot_s,depot_t) for t in sample_arcs.index.to_list()]
    return reformatted_arcs


In [4]:
class InitialRouteGenerator:
    def __init__(self, no_truck, no_dock, no_customer, customer_demand, constant_dict, distance_matrix):
        # CONSTANTs
        self.no_truck = no_truck
        self.no_dock = no_dock
        self.no_customer = no_customer
        self.customer_demand = customer_demand
        self.distance_matrix = distance_matrix
        self.truck_capacity = constant_dict['truck_capacity']
        self.fixed_setup_time = constant_dict['fixed_setup_time']

    def generateNodeSet(self):
        self.docking,self.customers,self.depot,self.depot_s,self.depot_t,self.all_depot,self.nodes,self.arcs=vis_sol.create_nodes(self.no_dock,self.no_customer)
        self.arcs = [','.join(list(l)) for l in self.arcs]
        self.arcs = self.splitDepotArcsVar(self.arcs,self.depot,self.depot_s,self.depot_t)
        return self.docking,self.customers,self.depot,self.depot_s,self.depot_t,self.all_depot,self.nodes,self.arcs  
    
    def generateArcs(self):
        #Not nesessary as arcs already generated with nodeSet
        nodes = set()
        nodes = list(nodes.union(set(self.docking),set(self.customers),set(self.depot)))
        no_node = len(nodes)
        node_combi = list(combinations(nodes,2))
        arc_permute = [list(permutations(list(c))) for c in node_combi ]
        sh = np.shape(arc_permute)
        arc_permute = np.reshape(arc_permute,(sh[0]*sh[1],sh[2])).tolist()
        arc_permute = [','.join(list(l)) for l in arc_permute]
        print(arc_permute)
        arc_permute = self.splitDepotArcsVar(arc_permute,self.depot,self.depot_s,self.depot_t)
        print("Finished creating truck_arcs:",len(arc_permute))
        return arc_permute
    
    def splitDepotArcsVar(self,a_var,depot,depot_s,depot_t):
        '''INPUT: ['depot,customer_1,T','customer_2,depot,T']
        OUTPUT:['depot_s,customer_1,T','customer_2,depot_t,T'] '''
        new_a_var =[]
        for a in a_var:
            v = a.split(',')
            if len(v)>1:
                if v[0]==depot[0]: v[0]=depot_s[0]
                if v[1]==depot[0]: v[1]=depot_t[0]
            new_a_var.append(','.join(v))
        return new_a_var
    
    def mergeDepotArcsVar(self,a_var,depot,depot_s,depot_t):
        '''INPUT: ['depot_s,customer_1,T','customer_2,depot_t,T']
        OUTPUT: ['depot,customer_1,T','customer_2,depot,T']'''
        new_a_var =[]
        for a in a_var:
            v = a.split(',')
            if len(v)>1:
                if v[0]==depot_s[0] or v[0]==depot_t[0]:v[0]=depot[0]
                if v[1]==depot_s[0] or v[1]==depot_t[0]:v[1]=depot[0]
            else:
                if v[0]==depot_s[0] or v[0]==depot_t[0]:v[0] = depot[0]
            new_a_var.append(','.join(v))
        return new_a_var    
    
    def generateAllCombiNodes(self,max_visited_nodes=None,str_node=None,end_node=None):
        '''THIS FUNCTION WILL GENERATE ALL POSSIBLE COMBINATIONS OF INPUT NODE LIST'''
        all_combi_nodes = []
        for i in range(1,max_visited_nodes+1):
            combination_set = list(combinations(self.customers,i))
            permutation_set = [list(permutations(list(c))) for c in combination_set ]
        #     print(permutation_set,np.shape(permutation_set))
            sh = np.shape(permutation_set)
        #     print(np.reshape(permutation_set,(sh[0]*sh[1],sh[2])))
            re_perm_set = np.reshape(permutation_set,(sh[0]*sh[1],sh[2])).tolist()
            all_combi_nodes = all_combi_nodes+re_perm_set
        # ADD depot 
        if (str_node is not None) and (end_node is not None):
            all_combi_nodes = [str_node+p+end_node for p in all_combi_nodes]
        else:all_combi_nodes = [p for p in all_combi_nodes]
        return all_combi_nodes
    
    def generateSetOfArcsFromNodesCombi(self,all_combi_nodes,added_type=''):
        '''THIS FUNCTION WILL GENERATE ARCS LIST BY CREATING BIGRAMS OF INPUT COMBI NODES''' 
        node2arcs = []
        for r in all_combi_nodes:    
            if len(r)==1: arc_list = [r]
            else: arc_list = list(nltk.bigrams(r))
            if added_type!='':arc_list = [','.join(list(a)+[added_type]) for a in arc_list]
            else:arc_list = [','.join(a) for a in arc_list]
            node2arcs.append(arc_list)
        return node2arcs
    
    def generateRoutes(self,initRouteDf,truck_cap_limit,
                       max_visited_nodes=None, max_vehicles_per_route=None, 
                       clustered=None, nbInitRoute=None,
                       mode='all',drone_cap_limit=None):
        if max_visited_nodes is None: max_visited_nodes=len(self.customers)
        if max_vehicles_per_route is None: max_vehicles_per_route=len(self.customers)
        self.all_combi_nodes=list()
        self.routes_arcs=list()
        t1 = time.time()
        _all_combi_nodes = self.generateAllCombiNodes(max_visited_nodes,self.depot_s,self.depot_t)
        self.all_combi_nodes+= _all_combi_nodes
        self.routes_arcs += self.generateSetOfArcsFromNodesCombi(_all_combi_nodes,'')
        
        if nbInitRoute is None: nbInitRoute = len(self.all_combi_nodes)
        print('nbInitRoute is set to',nbInitRoute)
        
        previous_cols = initRouteDf.columns[initRouteDf.columns.str.contains('r')].shape[0]
        ## ADD NEW COL TO DATAFRAME
        counter = 0
        for idx in range(len(self.all_combi_nodes)):
            if ((idx/len(self.all_combi_nodes))*100 % 10)==0: 
                print('progress:',idx,'/',len(self.all_combi_nodes))
            if idx>nbInitRoute: break
            lr_route = self.calculateLr(self.routes_arcs[idx])
            qr_route = pd.Series(self.all_combi_nodes[idx]).apply(lambda x: self.customer_demand[x]).sum()
            veh_min = int(np.ceil(qr_route*lr_route/self.truck_capacity))
            
            if veh_min > max_vehicles_per_route: continue
            else:
                for veh_no in range(veh_min,max_vehicles_per_route+1):
                    if veh_no < veh_min: continue
                    else:
                        self.addNewCol(initRouteDf, lr_route, veh_no,self.all_combi_nodes[idx],self.routes_arcs[idx],'route['+str(counter)+']')
                        counter += 1; #print(veh_no,veh_min)
        initRouteDf['labels']=init_path.splitDepotArcsVar(initRouteDf.labels,self.depot,[self.all_depot[0]],[self.all_depot[1]])
        self.init_routes_df = initRouteDf.copy()
#         self.init_routes_df = self.init_routes_df.set_index('labels')
        print('#Feasible Cols:', len(initRouteDf.columns)-1)
        print('Elased-time:',time.time()-t1)
    
    def generateBasicInitialPatterns(self,nbInitRoute,initRouteDf):
        columns = ['PathCoeff']
        path = pd.DataFrame(index = range(nbInitRoute), columns = columns)
        path['PathCoeff']=[initRouteDf[initRouteDf.columns[idx]].values for idx in range(nbInitRoute)]
        return path
    
    
    def addNewCol(self, df, col_cost, veh_no, nodes, arcs, var_name=None):
        # SET DEFAULT AS INDEXING
        coeff = nodes+arcs
        if var_name is None:
            var_name = df.columns.shape[0]
        df[var_name]= np.in1d(df.labels,coeff).astype(int)
        df.loc[df['labels']=='m',var_name] = veh_no
        df.loc[df['labels']=='lr',var_name] = col_cost
    
    
    def calculateLr(self, route_arcs):
        formatted_routes = self.mergeDepotArcsVar(route_arcs,self.depot,self.depot_s,self.depot_t)
        lr = self.fixed_setup_time+(pd.Series(formatted_routes).apply(lambda x:self.distance_matrix[tuple(x.split(','))]).sum()/20)
        return lr
    

### Phase_I Class

In [5]:
class phaseIModel:
    def __init__(self, _init_route, _initializer,
                 _distance_matrix,constant_dict,
                 extra_constr=None, _model_name = "PhaseI"):
        
        self.init_route = _init_route.copy()
        self.route_coeff = _init_route['PathCoeff'].values
        
        self.depot = _initializer.depot
        self.depot_s = _initializer.depot_s
        self.depot_t = _initializer.depot_t
        self.all_depot = _initializer.all_depot
        self.customers = _initializer.customers
        self.nodes = _initializer.nodes
        self.arcs = _initializer.arcs
    

        self.coeff_series = _initializer.init_routes_df['labels']
        self.depot_index = self.coeff_series[self.coeff_series.isin(self.all_depot)].index.values
        self.depot_s_index = self.coeff_series[self.coeff_series.isin(self.depot_s)].index.values
        self.depot_t_index = self.coeff_series[self.coeff_series.isin(self.depot_t)].index.values
        self.customer_index = self.coeff_series[self.coeff_series.isin(self.customers)].index.values
        self.nodes_index = self.coeff_series[self.coeff_series.isin(self.nodes)].index.values
        self.arcs_index = self.coeff_series[self.coeff_series.str.contains(',')].index.values
        self.veh_no_index = self.coeff_series.loc[self.coeff_series=='m'].index.values
        self.lr_index = self.coeff_series.loc[self.coeff_series=='lr'].index.values
    
        self.constant_dict = constant_dict
        self.vehicle_capacity = constant_dict['truck_capacity']
        self.fixed_setup_time = constant_dict['fixed_setup_time']
        self.truck_speed = constant_dict['truck_speed']
        self.distance_matrix = distance_matrix
        
        self.route_index = pd.Series(self.init_route.index).index.values
        self.model = Model(_model_name)

    def buildModel(self):
        self.generateVariables()
        self.generateConstraints()
        self.generateObjective()
        self.model.update()
        
    def generateVariables(self):
        self.route = self.model.addVars(self.route_index, lb=0,
                                       vtype=GRB.BINARY, name='route')
        print('Finish generating variables!')
        
    def generateConstraints(self):          
        const1 = ( quicksum(self.route_coeff[rt][i]*self.route[rt] for rt in self.route_index) ==1 \
                             for i in self.customer_index )
        self.model.addConstrs( const1,name='customer_coverage' )
        print('Finish generating constrains!')
        
    def generateObjective(self):
        # Minimize the total cost of the used rolls
        self.model.setObjective( quicksum(self.route[rt]*(self.route_coeff[rt][self.veh_no_index[0]]) for rt in self.route_index) ,
                                sense=GRB.MINIMIZE)
        print('Finish generating objective!')
        
    def solveModel(self, timeLimit = None,GAP=None):
        self.model.setParam('SolutionNumber',2)
        self.model.setParam(GRB.Param.PoolSearchMode, 2)
        self.model.setParam('TImeLimit', timeLimit)
        self.model.setParam('MIPGap',GAP)
        self.model.optimize()

### Phase_II Class

In [6]:
class phaseIIModel:
    def __init__(self, _init_route, _initializer,
                 _distance_matrix,constant_dict,
                 extra_constr=None, _model_name = "PhaseI"):
        
        self.init_route = _init_route.copy()
        self.route_coeff = _init_route['PathCoeff'].values
        self.init_routes_df = _initializer.init_routes_df
        
        self.depot = _initializer.depot
        self.depot_s = _initializer.depot_s
        self.depot_t = _initializer.depot_t
        self.all_depot = _initializer.all_depot
        self.customers = _initializer.customers
        self.nodes = _initializer.nodes
        self.arcs = _initializer.arcs
        self.route_cost = []
    

        self.coeff_series = _initializer.init_routes_df['labels']
        self.depot_index = self.coeff_series[self.coeff_series.isin(self.all_depot)].index.values
        self.depot_s_index = self.coeff_series[self.coeff_series.isin(self.depot_s)].index.values
        self.depot_t_index = self.coeff_series[self.coeff_series.isin(self.depot_t)].index.values
        self.customer_index = self.coeff_series[self.coeff_series.isin(self.customers)].index.values
        self.nodes_index = self.coeff_series[self.coeff_series.isin(self.nodes)].index.values
        self.arcs_index = self.coeff_series[self.coeff_series.str.contains(',')].index.values
        self.veh_no_index = self.coeff_series.loc[self.coeff_series=='m'].index.values
        self.lr_index = self.coeff_series.loc[self.coeff_series=='lr'].index.values
    
        self.constant_dict = constant_dict
        self.vehicle_capacity = constant_dict['truck_capacity']
        self.fixed_setup_time = constant_dict['fixed_setup_time']
        self.truck_speed = constant_dict['truck_speed']
        self.distance_matrix = distance_matrix
        self.customer_demand = _initializer.customer_demand
        self.max_vehicles = constant_dict['max_vehicles']
        
        self.route_index = pd.Series(self.init_route.index).index.values
        self.model = Model(_model_name)

    def buildModel(self):
        self.generateVariables()
        self.generateConstraints()
        self.generateCostOfRoutes()
        self.generateObjective()
        self.model.update()
        
    def generateVariables(self):
        self.route = self.model.addVars(self.route_index, lb=0,
                                       vtype=GRB.BINARY, name='route')
        print('Finish generating variables!')
        
    def generateConstraints(self):          
        const1 = ( quicksum(self.route_coeff[rt][i]*self.route[rt] for rt in self.route_index) >=1 \
                             for i in self.customer_index )
        self.model.addConstrs( const1,name='customer_coverage' )
        
        const2 = (quicksum(self.route[rt]*(self.route_coeff[rt][self.veh_no_index[0]]) for rt in self.route_index)==self.max_vehicles)
        self.model.addConstr(const2,name='vehicles_usage' )
        print('Finish generating constrains!')
    
    def generateCostOfRoutes(self):
        t1=time.time()
        self.route_cost = self.init_routes_df.set_index('labels').apply(lambda col: self.calculateCostOfRoute(col),axis=0)
        print('Finish generating cost vector!....Elapsed-time:',time.time()-t1)
    def calculateCostOfRoute(self, route):
        visiting_nodes = pd.Series(route[self.customer_index][route==1].index)
        visiting_arcs = pd.Series(route[self.arcs_index][route==1].index)
        next_node = ['depot_s']
#         print(route)
        route_cost = 0
        qr = visiting_nodes.apply(lambda x: self.customer_demand[x]).sum()
        avg_waiting = qr*route['lr']/(2*route['m'])
        while next_node[0]!='depot_t':
            selecting_node = next_node.pop(0)
#             print(selecting_node)
            arc = visiting_arcs[visiting_arcs.str.contains(selecting_node+',')].values[0].split(',')
            node_j = arc[1]
            next_node.append(node_j)
            qj = self.customer_demand[node_j]
            formatted_arc = tuple(merge_depot_arcs_var([','.join(arc)],self.depot,
                                                          self.depot_s,self.depot_t)[0].split(','))
            traveling_time_carrying_pkg = qr*(self.distance_matrix[formatted_arc])/self.constant_dict['truck_speed']
#             print(arc, qj, self.distance_matrix[formatted_arc])
#             print(traveling_time_carrying_pkg)
            route_cost+=traveling_time_carrying_pkg
            qr = qr-qj
    
#         print(avg_waiting,route_cost)
        route_cost += avg_waiting
#         print(route_cost)
        return route_cost
        
    def generateObjective(self):
        # Minimize the total cost of the used rolls
        self.model.setObjective( quicksum(self.route[rt]*(self.route_cost[rt]) for rt in self.route_index) ,
                                sense=GRB.MINIMIZE)
        print('Finish generating objective!')
        
    def solveModel(self, timeLimit = None,GAP=None):
        self.model.setParam('SolutionNumber',2)
        self.model.setParam(GRB.Param.PoolSearchMode, 2)
        self.model.setParam('TImeLimit', timeLimit)
        self.model.setParam('MIPGap',GAP)
        self.model.optimize()
        
    ##RELAXATION
    def solveRelaxedModel(self):
        #Relax integer variables to continous variables
        self.relaxedModel = self.model.relax()
        self.relaxedModel.optimize()
        
    def getRelaxSolution(self):
        a = pd.Series(self.relaxedModel.getAttr('X'))
        return a[a>0]

    def getDuals(self):
        return self.relaxedModel.getAttr('Pi',self.model.getConstrs())